In [0]:
# ── Silver: dim_clubs ───────────────────────────────────────────
from pyspark.sql import functions as F

bronze_clubs = spark.table("football_catalog.bronze.clubs")
target_table = "football_catalog.silver.dim_clubs"

print("--- Processing: dim_clubs ---")

# Clean and cast the core columns from the bronze clubs table
silver_clubs = (bronze_clubs
    .select(
        F.col("club_id").cast("integer"),
        F.trim(F.col("name")).alias("name"),
        F.col("domestic_competition_id"),
        F.col("squad_size").cast("integer"),
        F.trim(F.col("stadium_name")).alias("stadium_name")
    )
    # Data quality checks
    .filter(F.col("club_id").isNotNull())
    .dropDuplicates(["club_id"])
    # Add SCD flag so the Gold layer can filter for active records
    .withColumn("is_current", F.lit(True)) 
)

# Write to Silver layer
(silver_clubs.write
 .format("delta")
 .mode("overwrite") 
 .option("overwriteSchema", "true")
 .saveAsTable(target_table))

print(f"SUCCESS: Created {target_table}")
print(f"Total rows: {spark.table(target_table).count():,}")